In [1]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os
import json

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    api_key=groq_api_key
)

In [2]:
def get_video_id(url):
    if "v=" in url:
        video_id = url.split("v=")[1]
        video_id = video_id.split("&")[0]
        return video_id
    elif "youtu.be/" in url:
        video_id = url.split("youtu.be/")[1]
        video_id = video_id.split("?")[0]
        return video_id
    else:
        return None

def get_transcript(url):
    video_id = get_video_id(url)
    if not video_id:
        print("Invalid Youtube URL")
        return None
    try:
        ytt_api = YouTubeTranscriptApi()
        transcript = ytt_api.fetch(video_id)
        return transcript
    except Exception as e:
        print("Error: Could not fetch transcript. The video may not have captions or may not exist.")
        return None

def chunk_transcript(transcript, chunk_duration=120):
    chunks = []
    current_text = ""
    chunk_start = transcript[0].start

    for segment in transcript:
        if segment.start - chunk_start >= chunk_duration:
            chunks.append({
                "text": current_text.strip(),
                "start": chunk_start,
                "end": segment.start
            })
            current_text = segment.text + " "
            chunk_start = segment.start
        else:
            current_text += segment.text + " "

    # Don't forget the last chunk that's still being built
    if current_text:
        chunks.append({
            "text": current_text.strip(),
            "start": chunk_start,
            "end": transcript[-1].start
        })

    return chunks


class TranscriptionAgent:
    def __init__(self):
        self.name = "Transcription Agent"
    
    def run(self, url):
        transcript = get_transcript(url)
        if not transcript:
            return None
        chunks = chunk_transcript(transcript)
        return chunks

In [3]:
agent1 = TranscriptionAgent()
chunks = agent1.run("https://www.youtube.com/watch?v=aircAruvnKk")
print(f"Agent: {agent1.name}")
print(f"Chunks generated: {len(chunks)}")

Agent: Transcription Agent
Chunks generated: 10


In [4]:
class ConceptExtractionAgent:
    def __init__(self, llm):
        self.name = "Concept Extraction Agent"
        self.llm = llm
    
    def run(self, chunks):
        all_concepts = []
        
        for chunk in chunks:
            start_min = int(chunk["start"] // 60)
            start_sec = int(chunk["start"] % 60)
            end_min = int(chunk["end"] // 60)
            end_sec = int(chunk["end"] % 60)
            timestamp = f"{start_min}:{start_sec:02d} - {end_min}:{end_sec:02d}"
            
            prompt = f"""Extract the 2-3 most important concepts or topics from this transcript section.

Transcript:
{chunk['text']}

Return ONLY JSON in this format:
{{"concepts": ["concept 1", "concept 2", "concept 3"]}}
"""
            response = self.llm.invoke(prompt)
            content = response.content.strip()
            if content.startswith("```json"):
                content = content[7:]
            if content.startswith("```"):
                content = content[3:]
            if content.endswith("```"):
                content = content[:-3]
            content = content.strip()
        
            concepts = json.loads(content)
            concepts["timestamp"] = timestamp
            all_concepts.append(concepts)
        
        return all_concepts

In [5]:
agent2 = ConceptExtractionAgent(llm)
concepts = agent2.run(chunks)

for c in concepts:
    print(f"\n[{c['timestamp']}]")
    print(f"Concepts: {c['concepts']}")


[0:04 - 2:05]
Concepts: ['Neural Networks', 'Machine Learning', 'Image Recognition']

[2:05 - 4:06]
Concepts: ['Neural Networks', 'Neurons and Activations', 'Network Layers']

[4:06 - 6:07]
Concepts: ['Neural Network Structure', 'Digit Recognition', 'Layered Information Processing']

[6:07 - 8:08]
Concepts: ['Layered Neural Network Structure', 'Feature Detection and Recognition', 'Hierarchical Abstraction in Image Recognition']

[8:08 - 10:10]
Concepts: ['Neural Network Architecture', 'Weighted Sum', 'Feature Extraction']

[10:10 - 12:12]
Concepts: ['Sigmoid Function', 'Neural Network Weights and Biases', 'Activation of Neurons']

[12:12 - 14:14]
Concepts: ['Neural Network Architecture', 'Weights and Biases', 'Network Training and Learning']

[14:14 - 16:15]
Concepts: ['Linear Algebra', 'Neural Network Architecture', 'Matrix Vector Multiplication']

[16:15 - 18:15]
Concepts: ['Neural Network Training', 'Sigmoid Function', 'ReLU (Rectified Linear Unit)']

[18:15 - 18:25]
Concepts: ['Re

In [6]:
class QuestionGenerationAgent:
    def __init__(self, llm):
        self.name = "Question Generation Agent"
        self.llm = llm
    
    def run(self, concepts_list):
        all_questions = []
        
        for item in concepts_list:
            concepts = ", ".join(item["concepts"])
            timestamp = item["timestamp"]
            
            prompt = f"""Generate 1 multiple choice question that tests understanding of these concepts: {concepts}

The question should have:
- 4 options (A, B, C, D)
- The correct answer
- An explanation of why that answer is correct

Return ONLY JSON in this format:
{{
    "question": "the question text",
    "options": {{"A": "...", "B": "...", "C": "...", "D": "..."}},
    "correct_answer": "B",
    "explanation": "why this is correct"
}}
"""
            response = self.llm.invoke(prompt)
            content = response.content.strip()
            if content.startswith("```json"):
                content = content[7:]
            if content.startswith("```"):
                content = content[3:]
            if content.endswith("```"):
                content = content[:-3]
            content = content.strip()
            
            question = json.loads(content)
            question["timestamp"] = timestamp
            question["concepts"] = item["concepts"]
            all_questions.append(question)
        
        return {"questions": all_questions}

In [7]:
agent3 = QuestionGenerationAgent(llm)
quiz = agent3.run(concepts)

for q in quiz["questions"]:
    print(f"\n[{q['timestamp']}] Concepts: {q['concepts']}")
    print(f"Q: {q['question']}")


[0:04 - 2:05] Concepts: ['Neural Networks', 'Machine Learning', 'Image Recognition']
Q: What type of machine learning model is commonly used for image recognition tasks, such as classifying objects in images?

[2:05 - 4:06] Concepts: ['Neural Networks', 'Neurons and Activations', 'Network Layers']
Q: What is the primary purpose of the activation function in a neural network neuron?

[4:06 - 6:07] Concepts: ['Neural Network Structure', 'Digit Recognition', 'Layered Information Processing']
Q: In a neural network designed for digit recognition, which of the following best describes the role of the hidden layers in the network's structure?

[6:07 - 8:08] Concepts: ['Layered Neural Network Structure', 'Feature Detection and Recognition', 'Hierarchical Abstraction in Image Recognition']
Q: In a layered neural network structure for image recognition, what is the primary role of the early layers in the hierarchy?

[8:08 - 10:10] Concepts: ['Neural Network Architecture', 'Weighted Sum', 'Feat

In [10]:
class EvaluationAgent:
    def __init__(self, llm):
        self.name = "Evaluation Agent"
        self.llm = llm
    
    def run(self, quiz_data):
        score = 0
        weak_topics = []
        
        for i, q in enumerate(quiz_data["questions"]):
            print(f"\nQuestion {i+1}: {q['question']}")
            for option, text in q["options"].items():
                print(f"  {option}) {text}")
            user_answer = input("\nYour answer (A/B/C/D): ").upper()
            if user_answer == q['correct_answer']:
                score = score + 1
                print("Correct!")
            else:
                print("Incorrect!")
                weak_topics.extend(q['concepts'])
                weak_topics = list(set(weak_topics))
            print(f"Correct Answer: {q['correct_answer']}")
            print(f"Explanation: {q['explanation']}")
            print(f"Review in video at: {q['timestamp']}")
            
        print(f"\nFinal Score: {score}/{len(quiz_data['questions'])}")
        print(f"\nWeak Topics: {weak_topics}")

        return {
            "score": score,
            "total": len(quiz_data["questions"]),
            "weak_topics": weak_topics
        }

In [ ]:
# Agent 1: Get transcript
agent1 = TranscriptionAgent()
chunks = agent1.run("https://www.youtube.com/watch?v=aircAruvnKk")
print(f"Agent 1 done: {len(chunks)} chunks")

# Agent 2: Extract concepts
agent2 = ConceptExtractionAgent(llm)
concepts = agent2.run(chunks)
print(f"Agent 2 done: {len(concepts)} concept sets")

# Agent 3: Generate questions
agent3 = QuestionGenerationAgent(llm)
quiz = agent3.run(concepts)
print(f"Agent 3 done: {len(quiz['questions'])} questions")

# Agent 4: Evaluate
agent4 = EvaluationAgent(llm)
results = agent4.run(quiz)

Agent 1 done: 10 chunks
Agent 2 done: 10 concept sets
Agent 3 done: 10 questions

Question 1: What type of machine learning model is commonly used for image recognition tasks, such as classifying objects in images?
  A) Linear Regression
  B) Convolutional Neural Networks
  C) Decision Trees
  D) Support Vector Machines



Your answer (A/B/C/D):  a


Incorrect!
Correct Answer: B
Explanation: Convolutional Neural Networks (CNNs) are a type of neural network specifically designed for image recognition tasks. They use convolutional and pooling layers to extract features from images, allowing them to learn complex patterns and classify objects with high accuracy. This makes them the most suitable choice for image recognition tasks.
Review in video at: 0:04 - 2:05

Question 2: What is the primary function of the activation function in a neural network neuron?
  A) To reduce the dimensionality of the input data
  B) To introduce non-linearity into the neuron's output
  C) To increase the number of layers in the network
  D) To decrease the number of neurons in a layer



Your answer (A/B/C/D):  a


Incorrect!
Correct Answer: B
Explanation: The activation function is used to introduce non-linearity into the neuron's output, allowing the neural network to learn and represent more complex relationships between inputs and outputs. Without an activation function, the neuron's output would be a linear combination of its inputs, limiting the network's ability to learn and generalize.
Review in video at: 2:05 - 4:06

Question 3: What is the primary function of the hidden layers in a neural network used for digit recognition and pattern analysis?
  A) To accept user input and display the output
  B) To apply non-linear transformations to the input data, allowing the network to learn complex patterns
  C) To store the network's weights and biases
  D) To perform the final classification or prediction


...

Incorrect!
Correct Answer: B
Explanation: The hidden layers in a neural network are responsible for applying non-linear transformations to the input data, allowing the network to learn complex patterns and relationships. This is particularly important in digit recognition and pattern analysis tasks, where the network needs to extract features and make decisions based on nuanced patterns in the input data. The correct answer, option B, reflects this understanding of the role of hidden layers in neural networks.
Review in video at: 4:06 - 6:07

Question 4: In a neural network, what is the primary function of the early layers in terms of feature detection and hierarchical abstraction?
  A) To detect high-level abstract features such as objects
  B) To detect low-level features such as edges and lines, which are then combined into higher-level features in later layers
  C) To perform classification tasks directly without any intermediate feature detection
  D) To reduce the dimensionality 


Your answer (A/B/C/D):  a


Incorrect!
Correct Answer: B
Explanation: The early layers in a neural network are responsible for detecting low-level features such as edges, lines, and textures. These features are then combined in later layers to form higher-level features, such as shapes and objects, through a process known as hierarchical abstraction. This process allows the network to learn complex and abstract representations of the input data, enabling it to perform tasks such as image classification and object recognition.
Review in video at: 6:07 - 8:08

Question 5: In a neural network architecture, what is the primary purpose of the weighted sum in the context of feature extraction?
  A) To reduce the dimensionality of the input data
  B) To combine the features extracted by each neuron to produce an output
  C) To apply a non-linear transformation to the input data
  D) To normalize the input data



Your answer (A/B/C/D):  a


Incorrect!
Correct Answer: B
Explanation: The weighted sum is used to combine the features extracted by each neuron, where each feature is multiplied by a weight that represents its importance, and the results are summed to produce the output of the neuron. This allows the neural network to learn complex representations of the input data by weighting the importance of different features.
Review in video at: 8:08 - 10:10

Question 6: What is the primary purpose of the sigmoid function in a neural network?
  A) To increase the complexity of the model by adding more weights and biases
  B) To introduce non-linearity into the model, allowing it to learn and represent more complex relationships between inputs and outputs
  C) To normalize the inputs to have zero mean and unit variance
  D) To reduce overfitting by penalizing large weights and biases
